# Network Anomaly Detection - ML-T2-002

## Detecting Previously Unseen Network Attacks

**Goal:** Train models on normal network traffic only, then detect anomalies representing unseen attacks.

**Approach:**
1. Data preprocessing and exploration
2. Baseline models: Isolation Forest and One-Class SVM
3. Advanced model: Autoencoder (Deep Learning)
4. Model comparison and evaluation

---

In [ ]:
# Import libraries
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Data Preparation

We'll use synthetic data for demonstration. In production, replace this with CICIDS2017 or UNSW-NB15 dataset.

In [ ]:
from data_preprocessing import generate_sample_data, NetworkDataPreprocessor

# Generate synthetic network traffic data
print("Generating synthetic network traffic data...\n")
X_train, X_test, y_train, y_test = generate_sample_data(
    n_samples=10000,
    n_features=20,
    contamination=0.1  # 10% attacks in the full dataset
)

print(f"\n✓ Data prepared successfully!")
print(f"  Training samples: {X_train.shape}")
print(f"  Test samples: {X_test.shape}")
print(f"  Test attacks: {y_test.sum()} / {len(y_test)} ({y_test.sum()/len(y_test)*100:.1f}%)")

### Alternative: Load Real Dataset

Uncomment the code below to load CICIDS2017 or UNSW-NB15 dataset:

In [ ]:
# # Load real dataset
# preprocessor = NetworkDataPreprocessor(dataset_type='UNSW-NB15')
# X_train, X_test, y_train, y_test = preprocessor.preprocess_pipeline(
#     filepath='../data/UNSW-NB15.csv',
#     label_column='label',
#     normal_label='Normal'
# )

## 2. Baseline Models

### 2.1 Isolation Forest

In [ ]:
from baseline_models import BaselineAnomalyDetector

# Initialize detector
detector = BaselineAnomalyDetector()

# Train Isolation Forest
if_model = detector.train_isolation_forest(
    X_train,
    contamination=0.1,
    random_state=42
)

# Evaluate
if_results = detector.evaluate('isolation_forest', X_test, y_test)

### 2.2 One-Class SVM

In [ ]:
# Train One-Class SVM
svm_model = detector.train_one_class_svm(
    X_train,
    nu=0.1,
    kernel='rbf'
)

# Evaluate
svm_results = detector.evaluate('one_class_svm', X_test, y_test)

### 2.3 Baseline Model Comparison

In [ ]:
# Compare models
comparison_df = detector.compare_models()
print("\nBaseline Model Comparison:")
print(comparison_df)

## 3. Advanced Model: Autoencoder

### 3.1 Build and Train Autoencoder

In [ ]:
from autoencoder_model import AutoencoderAnomalyDetector

# Initialize autoencoder
ae_detector = AutoencoderAnomalyDetector(input_dim=X_train.shape[1])

# Build model
ae_detector.build_model(
    layers=[32, 16, 8, 16, 32],
    dropout_rate=0.1,
    learning_rate=0.001
)

In [ ]:
# Train autoencoder
history = ae_detector.train(
    X_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    patience=5,
    model_dir='../models'
)

### 3.2 Training History Visualization

In [ ]:
# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Autoencoder Training History', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Evaluate Autoencoder

In [ ]:
# Evaluate autoencoder
ae_results = ae_detector.evaluate(X_test, y_test)

### 3.4 Reconstruction Error Distribution

In [ ]:
# Plot reconstruction error distribution
ae_detector.plot_reconstruction_distribution(X_test, y_test)

## 4. Model Comparison

### 4.1 Performance Metrics

In [ ]:
# Compile all results
all_results = pd.DataFrame([
    {
        'Model': 'Isolation Forest',
        'Precision': if_results['precision'],
        'Recall': if_results['recall'],
        'F1-Score': if_results['f1_score'],
        'ROC-AUC': if_results['roc_auc']
    },
    {
        'Model': 'One-Class SVM',
        'Precision': svm_results['precision'],
        'Recall': svm_results['recall'],
        'F1-Score': svm_results['f1_score'],
        'ROC-AUC': svm_results['roc_auc']
    },
    {
        'Model': 'Autoencoder',
        'Precision': ae_results['precision'],
        'Recall': ae_results['recall'],
        'F1-Score': ae_results['f1_score'],
        'ROC-AUC': ae_results['roc_auc']
    }
])

print("\n" + "="*70)
print("Final Model Comparison")
print("="*70)
print(all_results.to_string(index=False))

# Find best model
best_model_idx = all_results['F1-Score'].idxmax()
best_model = all_results.iloc[best_model_idx]['Model']
print(f"\n✓ Best Model (by F1-Score): {best_model}")

### 4.2 Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Isolation Forest
sns.heatmap(
    if_results['confusion_matrix'],
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal', 'Attack'],
    yticklabels=['Normal', 'Attack'],
    ax=axes[0]
)
axes[0].set_title('Isolation Forest', fontsize=14)
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# One-Class SVM
sns.heatmap(
    svm_results['confusion_matrix'],
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal', 'Attack'],
    yticklabels=['Normal', 'Attack'],
    ax=axes[1]
)
axes[1].set_title('One-Class SVM', fontsize=14)
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_xlabel('Predicted Label', fontsize=12)

# Autoencoder
sns.heatmap(
    ae_results['confusion_matrix'],
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal', 'Attack'],
    yticklabels=['Normal', 'Attack'],
    ax=axes[2]
)
axes[2].set_title('Autoencoder', fontsize=14)
axes[2].set_ylabel('True Label', fontsize=12)
axes[2].set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout()
plt.show()

### 4.3 ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc

# Calculate ROC curves
plt.figure(figsize=(10, 8))

# Isolation Forest
fpr_if, tpr_if, _ = roc_curve(y_test, if_results['scores'])
plt.plot(fpr_if, tpr_if, label=f"Isolation Forest (AUC = {if_results['roc_auc']:.3f})", linewidth=2)

# One-Class SVM
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_results['scores'])
plt.plot(fpr_svm, tpr_svm, label=f"One-Class SVM (AUC = {svm_results['roc_auc']:.3f})", linewidth=2)

# Autoencoder
fpr_ae, tpr_ae, _ = roc_curve(y_test, ae_results['scores'])
plt.plot(fpr_ae, tpr_ae, label=f"Autoencoder (AUC = {ae_results['roc_auc']:.3f})", linewidth=2)

# Random baseline
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - All Models', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.4 Performance Bar Chart

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    all_results.plot(x='Model', y=metric, kind='bar', ax=ax, legend=False, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_title(metric, fontsize=14)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(all_results['Model'], rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Save Models

In [ ]:
# Save trained models
os.makedirs('../models', exist_ok=True)

detector.save_model('isolation_forest', '../models/isolation_forest.pkl')
detector.save_model('one_class_svm', '../models/one_class_svm.pkl')
ae_detector.save_model('../models/autoencoder_best.keras')

print("\n✓ All models saved successfully!")

## 6. Key Findings

### Summary:
- **Isolation Forest**: Fast training, good for high-dimensional data, decent performance
- **One-Class SVM**: More computationally expensive, works well with smaller datasets
- **Autoencoder**: Best reconstruction-based approach, captures complex patterns in normal traffic

### Recommendations:
1. For real-time detection: Use Isolation Forest (fastest inference)
2. For highest accuracy: Use Autoencoder (best at learning normal patterns)
3. For balanced approach: Ensemble of all three models

### Next Steps:
1. Test on real network traffic datasets (CICIDS2017, UNSW-NB15)
2. Deploy best model in Streamlit dashboard
3. Implement ensemble approach for production
4. Set up continuous monitoring and retraining pipeline

---
## Project Complete! ✓

**Next:** Run the Streamlit dashboard to interact with the models:
```bash
cd ../streamlit_app
streamlit run app.py
```